# Ранжирование русскоязычных фрагментов

Исторический запуск обучения и инференса выполнен на Windows x64, Python 3.10.11, NVIDIA RTX 5070 Ti 16 ГБ, PyTorch 2.8.0 с CUDA 12.8.

Ноутбук сохранит результат в `submission.csv`.

Ноутбук получает веса из Hugging Face, строит обучающие разбиения, обучает модели и сохраняет прогнозы в порядке образца.

In [ ]:
from contextlib import contextmanager
from datetime import datetime, timezone
from importlib import metadata
import json
import os
from pathlib import Path
import platform
import sys
import tempfile

import numpy as np
import pandas as pd

DATA_FILES = ("train.csv", "test.csv", "sample_submission.csv")

def release_paths():
    data = Path.cwd()
    for filename in (globals().get("__vsc_ipynb_file__"), os.environ.get("JPY_SESSION_NAME")):
        if filename and Path(filename).suffix.lower() == ".ipynb" and Path(filename).is_file():
            data = Path(filename).resolve().parent
            break
    missing = [name for name in DATA_FILES if not (data / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Положите {', '.join(missing)} рядом с solution.ipynb. Папка: {data}")

    if os.environ.get("AMML_WORK_DIR"):
        return data, Path(os.environ["AMML_WORK_DIR"]).expanduser().resolve()
    workdirs = [folder / "outputs" / "ranking_v47" for folder in (data, Path.cwd(), data.parent)]
    existing = list(dict.fromkeys(work.resolve() for work in workdirs if (work / "upstream").is_dir()))
    if len(existing) > 1:
        raise ValueError(f"Найдено несколько папок прогона {existing}. Укажите нужную в AMML_WORK_DIR.")
    return data, existing[0] if existing else (data / "outputs" / "ranking_v47").resolve()

@contextmanager
def release_lock(directory):
    directory.mkdir(exist_ok=True, parents=True)
    with (directory / ".run.lock").open("a+b") as lock:
        lock.seek(0, 2)
        if lock.tell() == 0:
            lock.write(b"0")
            lock.flush()
        lock.seek(0)
        if os.name == "nt":
            import msvcrt
            acquire = lambda: msvcrt.locking(lock.fileno(), msvcrt.LK_NBLCK, 1)
            unlock = lambda: msvcrt.locking(lock.fileno(), msvcrt.LK_UNLCK, 1)
        else:
            import fcntl
            acquire = lambda: fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
            unlock = lambda: fcntl.flock(lock.fileno(), fcntl.LOCK_UN)
        try:
            acquire()
        except OSError as exc:
            raise RuntimeError(f"В каталоге {directory} уже идёт другой запуск.") from exc
        try:
            yield
        finally:
            lock.seek(0)
            unlock()

def release_atomic(path, payload):
    with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as temp:
        tmp = Path(temp.name)
        try:
            temp.write(payload)
            temp.flush()
            os.fsync(temp.fileno())
        except BaseException:
            temp.close()
            tmp.unlink(missing_ok=True)
            raise
    try:
        os.replace(tmp, path)
    finally:
        tmp.unlink(missing_ok=True)

def release_environment():
    import torch
    versions = {name: metadata.version(name) for name in (
        "torch", "numpy", "pandas", "transformers", "tokenizers", "peft", "bitsandbytes", "accelerate",
        "lightgbm", "catboost", "scikit-learn", "safetensors", "scipy", "huggingface-hub")}
    reference = {"torch":"2.8.0+cu128", "transformers":"4.57.1", "peft":"0.18.0",
                 "bitsandbytes":"0.48.2", "numpy":"2.2.6", "pandas":"2.2.3", "catboost":"1.2.8",
                 "lightgbm":"4.6.0", "scikit-learn":"1.6.1", "scipy":"1.15.3", "accelerate":"1.11.0"}
    differences = [
        f"{name}: установлена {versions[name]}, в requirements.txt {expected}"
        for name, expected in reference.items() if versions[name] != expected
    ]
    if differences:
        print("Предупреждение: версии библиотек отличаются. Продолжаем запуск.", *differences, sep="\n", flush=True)
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError("Для полного обучения нужна NVIDIA CUDA с аппаратным BF16.")
    return {"python": sys.version.split()[0], "system": platform.platform(), "gpu": torch.cuda.get_device_name(),
            "cuda": torch.version.cuda, "packages": versions}

def release_submission(test, sample, baseline, jina):
    if not baseline[["id", "query_id"]].reset_index(drop=True).equals(test[["id", "query_id"]].reset_index(drop=True)):
        raise ValueError("Порядок id и query_id базовой ветки расходится с test.csv.")
    if not test.id.is_unique or not sample.id.is_unique or set(test.id) != set(sample.id):
        raise ValueError("Ожидается взаимно однозначное соответствие ID исходному образцу.")
    groups = test.query_id.astype(str).to_numpy()
    def query_z(values):
        v = np.asarray(values, dtype=np.float64)
        if v.shape != (len(test),) or not np.isfinite(v).all():
            raise ValueError("Размер или значения выходов модели некорректны.")
        grouped = pd.Series(v).groupby(groups, sort=False)
        mean = grouped.transform("mean").to_numpy()
        sigma = grouped.transform("std", ddof=0).to_numpy()
        return (v - mean) / np.where(sigma > 1e-12, sigma, 1.)
    mixed = .25 * query_z(-baseline.v35.to_numpy()) + .75 * query_z(jina)
    ordered = test[["id", "query_id"]].assign(score=mixed).sort_values(
        ["query_id", "score", "id"], ascending=[True, False, True])
    ordered["prediction"] = -(ordered.groupby("query_id", sort=False).cumcount() + 1)
    result = sample[["id"]].join(ordered.set_index("id").prediction, on="id", validate="one_to_one")
    if result.prediction.isna().any():
        raise ValueError("Отсутствуют тестовые ранги.")
    # В историческом CSV Windows использовала CRLF. Задаём окончания строк явно.
    return result.to_csv(index=False, lineterminator="\r\n").encode("utf-8")


## BGE, LightGBM и Qwen

Пять исходных разбиений по компонентам запросов и документов дают оценки для метамоделей. Для теста базовые модели обучаются на всём train. При повторном запуске пригодные контрольные точки загружаются из рабочей папки.

In [ ]:
from pathlib import Path
from importlib.metadata import version as package_version
import gc
import hashlib
import html
import inspect
import json
import math
import os
import pickle
import random
import re
import time
from uuid import uuid4
from zipfile import BadZipFile

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import BitsAndBytesConfig, Qwen3ForCausalLM, get_linear_schedule_with_warmup

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

up_MODEL_ID = 'BAAI/bge-reranker-v2-m3'
up_MODEL_REVISION = '953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e'
up_SEED = 2026
up_MAX_LENGTH = 512
up_MICROBATCH = 2
up_ACCUMULATION = 16
up_INFER_BATCH = 4
up_EPOCHS = 3  # исходный scheduler horizon; не фактический бюджет fit
up_BGE_FRACTION = 0.75
up_LGB_ITERATIONS = 170
up_LEXICAL_WEIGHT = 0.10
up_LEARNING_RATE = 1e-5
up_WEIGHT_DECAY = 0.01
up_WARMUP_RATIO = 0.1
up_THREADS = 12
up_N_FOLDS = 5
up_QLORA_MICROBATCH = 1
up_QLORA_ACCUMULATION = 32
up_QLORA_LR = 5e-5
up_QLORA_RANK = 8
up_QLORA_ALPHA = 16

up_CHECKPOINT_FORMAT = 'amml-upstream-v2'
up_SCORE_COLUMNS = ('bge', 'lexical', 'raw_attention_0.50', 'raw_qwen_8b_0.50')
up_DEVICE = torch.device('cuda')

def up_seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    # SDPA/CUDA могут давать небольшие межплатформенные расхождения; фиксируем seed,
    # но не обещаем битовую идентичность между GPU/драйверами.

up_CSV_NAMES = ('train.csv', 'test.csv', 'sample_submission.csv')
up_BASE_COLUMNS = ['id', 'query_id', 'document_id', 'article_id', 'query', 'title', 'text',
                'passage_index', 'article_passage_count', 'article_char_count', 'ingest_source']
def upstream_read_inputs(data_dir):
    data_dir = Path(data_dir)
    missing_csv = [name for name in up_CSV_NAMES if not (data_dir / name).is_file()]
    if missing_csv:
        raise FileNotFoundError(f'В {data_dir.resolve()} не найдены {missing_csv}. Запустите Jupyter из папки с CSV.')
    train = pd.read_csv(data_dir / 'train.csv', dtype=str, keep_default_na=False)
    test = pd.read_csv(data_dir / 'test.csv', dtype=str, keep_default_na=False)
    sample = pd.read_csv(data_dir / 'sample_submission.csv', dtype=str, keep_default_na=False)
    if (list(train.columns) != up_BASE_COLUMNS + ['relevance']
            or list(test.columns) != up_BASE_COLUMNS or list(sample.columns) != ['id', 'prediction']):
        raise ValueError('Столбцы train.csv, test.csv или sample_submission.csv не соответствуют схеме.')
    for frame, name in ((train, 'train.csv'), (test, 'test.csv')):
        if frame.empty or not frame.id.is_unique or frame[['id', 'query_id', 'document_id']].eq('').any().any():
            raise ValueError(f'В {name} отсутствуют строки либо есть пустые или повторяющиеся идентификаторы строк.')
        if frame.groupby('query_id')['query'].nunique().max() != 1:
            raise ValueError(f'В {name} одному query_id соответствуют разные тексты запроса.')
    if not sample.id.is_unique or set(sample.id) != set(test.id):
        raise ValueError('ID образца отправки должны образовывать взаимно однозначное соответствие с test.csv.')
    if not train.relevance.isin(['0', '1']).all():
        raise ValueError('Допустимые метки relevance в train.csv — 0 и 1.')
    train['relevance'] = train.relevance.astype('int8')
    if not set(train.query_id).isdisjoint(test.query_id) or not set(train.document_id).isdisjoint(test.document_id):
        raise ValueError('Запросы и документы train и test должны быть непересекающимися.')
    return train, test, sample


def up_build_geometry(train):
    # DSU: только train query/document geometry, без labels.
    parent = {}
    def find(node):
        parent.setdefault(node, node)
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node
    def union(a, b):
        a, b = find(a), find(b)
        if a != b: parent[b] = a
    for q, d in train[['query_id', 'document_id']].itertuples(index=False, name=None):
        union(('query', q), ('document', d))
    query_component = {q: find(('query', q)) for q in sorted(train.query_id.unique())}
    geometry = train[['id', 'query_id', 'document_id']].copy()
    geometry['component'] = geometry.query_id.map(query_component).map(lambda x: x[1])
    components = geometry.groupby('component', sort=False).agg(queries=('query_id','nunique'), rows=('id','size'))
    components = components.sort_values(['queries','rows'], ascending=False, kind='stable')
    fold_totals = [{'queries':0,'rows':0} for _ in range(up_N_FOLDS)]
    component_fold = {}
    for c, row in components.iterrows():
        f = min(range(up_N_FOLDS), key=lambda i: (fold_totals[i]['queries'], fold_totals[i]['rows'], i))
        component_fold[c] = f
        for k in ('queries','rows'): fold_totals[f][k] += int(row[k])
    geometry['fold'] = geometry.component.map(component_fold)
    assert geometry.groupby('query_id').fold.nunique().max() == 1
    assert geometry.groupby('document_id').fold.nunique().max() == 1
    assert geometry.fold.notna().all() and len(geometry) == len(train)
    if set(geometry.fold) != set(range(up_N_FOLDS)):
        raise ValueError(f'Нужно минимум {up_N_FOLDS} непустых компонент для разбиения train.')
    return geometry


In [ ]:
up_TITLE_MISSING = {'', 'nan', 'none', 'null', 'нет заголовка', 'без заголовка', '[нет заголовка]', '[без заголовка]', 'заголовок отсутствует', 'n/a', 'na', 'unknown', 'неизвестно', 'нет данных', '[missing]', '[unk]'}
def up_clean_text(value):
    text = re.sub(r'</?[A-Za-z][^>]*>', ' ', html.unescape(str(value)))
    text = re.sub(r'\s+', ' ', text.replace('\u00a0', ' ').replace('\u202f', ' ')).strip()
    # Только точные повторы фразы навигации у начала строки; одиночную не удаляем.
    text = re.sub(r'^(?:(Главная\s*[»>|/]\s*){2,})', '', text, flags=re.I).strip()
    return text
def up_lexical(s): return str(s).lower().replace('ё', 'е')
def up_tokens(s): return re.findall(r'(?u)\b\w+\b', up_lexical(s))
def up_numeric(s):
    s = str(s).strip().replace('\u00a0',' ').replace('\u202f',' ')
    # Пробелы допустимы только как группы тысяч; единственный ',' — decimal separator.
    if not re.fullmatch(r'[+-]?(?:\d+|\d{1,3}(?: \d{3})+)(?:[.,]\d+)?', s): return np.nan
    try:
        v = float(s.replace(' ','').replace(',','.'))
        return v if np.isfinite(v) else np.nan
    except ValueError: return np.nan
def up_prepare(d):
    d = d.copy()
    for col in ('query','title','text'): d['clean_' + col] = d[col].map(up_clean_text)
    d['title_missing'] = d.clean_title.map(lambda t: up_lexical(t) in up_TITLE_MISSING or bool(re.fullmatch(r'[-—?]+', t))).astype('int8')
    d.loc[d.title_missing.eq(1), 'clean_title'] = ''
    d['pair_text'] = 'title: ' + d.clean_title + ' passage: ' + d.clean_text
    for col in ('passage_index','article_passage_count','article_char_count'):
        raw = d[col].map(up_numeric)
        invalid = raw.isna() | (raw < 0) | (raw.mod(1).fillna(0) != 0)
        if col != 'passage_index': invalid |= raw <= 0
        d[col + '_raw_numeric'] = raw
        d[col + '_invalid'] = invalid.astype('int8')
        d[col + '_valid'] = raw.mask(invalid)
    return d

class up_LexicalFeatures:
    def fit(self, d):
        unique = d.drop_duplicates('document_id')
        corpus = (unique.clean_title + ' ' + unique.clean_text).map(up_lexical).tolist()
        self.word = TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=100000,
                                    sublinear_tf=True, dtype=np.float32)
        self.char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=2,
                                    max_features=150000, sublinear_tf=True, dtype=np.float32)
        self.word.fit(corpus); self.char.fit(corpus)
        self.count = CountVectorizer(vocabulary=self.word.vocabulary_, ngram_range=(1,2), dtype=np.float32)
        counts = self.count.transform(corpus)
        self.avgdl = max(float(np.asarray(counts.sum(axis=1)).mean()), 1.0)
        df = np.asarray((counts > 0).sum(axis=0)).ravel()
        self.bm_idf = np.log1p((len(corpus) - df + .5) / (df + .5)).astype(np.float32)
        self.sources = sorted(d.ingest_source.unique())
        self.columns = None
        return self
    def transform(self, d):
        out = {}
        queries = d.clean_query.map(up_lexical).tolist()
        qw, qc = self.word.transform(queries), self.char.transform(queries)
        qcount = self.count.transform(queries)
        qbinary = qcount.copy(); qbinary.data[:] = 1
        for field in ('title','text'):
            docs = d['clean_' + field].map(up_lexical).tolist()
            out['word_cos_' + field] = np.asarray(qw.multiply(self.word.transform(docs)).sum(axis=1)).ravel()
            out['char_cos_' + field] = np.asarray(qc.multiply(self.char.transform(docs)).sum(axis=1)).ravel()
            cnt = self.count.transform(docs).tocsr()
            length = np.asarray(cnt.sum(axis=1)).ravel()
            norm = 1.2 * (1 - .75 + .75 * length / self.avgdl)
            bm = cnt.copy()
            bm.data = (bm.data * 2.2 / (bm.data + np.repeat(norm, np.diff(bm.indptr))))
            bm = bm.multiply(self.bm_idf)
            out['bm25_' + field] = np.asarray(qbinary.multiply(bm).sum(axis=1)).ravel()
            matching = []
            for q, text in zip(queries, docs):
                qt, dt = set(up_tokens(q)), set(up_tokens(text))
                qnums, dnums = set(re.findall(r'\d+',q)), set(re.findall(r'\d+',text))
                matching.append((len(qt & dt) / max(1,len(qt)), len(qt & dt) / max(1,len(dt)),
                                 float(bool(q) and q in text), len(qnums & dnums) / max(1,len(qnums)),
                                 len(text), len(up_tokens(text)), len(text) / max(1,len(q))))
            a = np.asarray(matching, dtype=np.float64)
            for j, name in enumerate(('coverage','jaccard_doc','phrase','numbers','chars','tokens','length_ratio')):
                out[field + '_' + name] = a[:,j]
        out['query_chars'] = d.clean_query.str.len().to_numpy()
        out['query_tokens'] = d.clean_query.map(lambda s: len(up_tokens(s))).to_numpy()
        out['title_missing'] = d.title_missing.to_numpy()
        for c in ('passage_index','article_passage_count','article_char_count'):
            for suffix in ('_raw_numeric','_invalid','_valid'): out[c + suffix] = d[c + suffix].to_numpy()
            out[c + '_log1p'] = np.log1p(d[c + '_valid'].to_numpy())
        idx = d.passage_index_valid.to_numpy(); count = d.article_passage_count_valid.to_numpy()
        chars = d.article_char_count_valid.to_numpy()
        out['index_over_count'] = idx / count
        out['index_gt_count'] = (idx > count).astype(float)
        out['char_count_below_text'] = (chars < d.clean_text.str.len().to_numpy()).astype(float)
        out['chars_over_passages'] = chars / count
        for i, source in enumerate(self.sources): out['source_' + str(i)] = d.ingest_source.eq(source).to_numpy(dtype=float)
        out['source_unknown'] = (~d.ingest_source.isin(self.sources)).to_numpy(dtype=float)
        out['html_flag'] = d.text.str.contains(r'</?[A-Za-z][^>]*>|&\w+;', regex=True).to_numpy(dtype=float)
        out['text_repeated_spaces'] = d.text.str.contains(r'\s{2,}', regex=True).to_numpy(dtype=float)
        X = pd.DataFrame(out).replace([np.inf,-np.inf], np.nan).astype('float32')
        if self.columns is None: self.columns = list(X.columns)
        assert list(X.columns) == self.columns and not any(c in X for c in up_BASE_COLUMNS[:4])
        return X

def up_sorted_for_lgb(d):
    return d.sort_values(['query_id','id'], kind='stable').reset_index(drop=True)
def up_lgb_group_sizes(d): return d.groupby('query_id', sort=False).size().to_numpy()
up_LGB_PARAMS = dict(objective='lambdarank', metric='None', label_gain=[0,1],
                  learning_rate=.035, num_leaves=15, min_data_in_leaf=40,
                  lambda_l2=5.0, feature_fraction=.9, verbosity=-1,
                  num_threads=up_THREADS, seed=up_SEED, deterministic=True, force_col_wise=True)
def up_fit_lexical(fit):
    import lightgbm as lgb

    ordered = up_sorted_for_lgb(fit)
    features = up_LexicalFeatures().fit(ordered)
    model = lgb.train(
        up_LGB_PARAMS,
        lgb.Dataset(features.transform(ordered), label=ordered.relevance.to_numpy(),
                    group=up_lgb_group_sizes(ordered)),
        num_boost_round=up_LGB_ITERATIONS,
    )
    return features, model



In [ ]:

def up_ordered_id_hash(frame):
    return up_fingerprint(frame.id.tolist())


def up_fingerprint(*values):
    payload = json.dumps(values, sort_keys=True).encode()
    return hashlib.sha256(payload).hexdigest()


def up_atomic_artifact(path, write):
    # A completed write replaces the current file on one filesystem. During the
    # rotation the preceding file is still available under its backup name.
    pending = path.with_name(f'.{path.name}.{uuid4().hex}.tmp')
    try:
        with pending.open('wb') as stream:
            write(stream)
            stream.flush()
            os.fsync(stream.fileno())
        if path.exists():
            os.replace(path, path.with_suffix(path.suffix + '.previous'))
        os.replace(pending, path)
    finally:
        pending.unlink(missing_ok=True)


def up_load_artifact(path, read):
    previous = path.with_suffix(path.suffix + '.previous')
    for candidate in (path, previous):
        if not candidate.is_file():
            continue
        try:
            restored = read(candidate)
        except (RuntimeError, OSError, EOFError, ValueError, KeyError, BadZipFile, pickle.UnpicklingError) as exc:
            print(f'Не удалось прочитать контрольную точку {candidate.name} ({type(exc).__name__}).')
            continue
        if candidate == previous:
            path.unlink(missing_ok=True)
            print(f'Восстановлена предыдущая точка {previous.name}.')
        return restored
    return None


def up_save_scores(path, signature, scores):
    values = np.asarray(scores, dtype=np.float64)
    if values.ndim != 1 or not np.isfinite(values).all():
        raise ValueError(f'Неконечные или неполные оценки в {path.name}.')
    up_atomic_artifact(path, lambda stream: np.savez_compressed(stream, signature=signature, scores=values))


def up_load_scores(path, signature, rows, batch_size):
    def read(candidate):
        with np.load(candidate, allow_pickle=False) as stored:
            if stored['signature'].item() != signature:
                raise ValueError('Параметры сохранённого инференса изменились.')
            scores = stored['scores']
        if (scores.dtype != np.float64 or scores.ndim != 1 or not 0 < len(scores) <= rows
                or (len(scores) != rows and len(scores) % batch_size) or not np.isfinite(scores).all()):
            raise ValueError('Сохранённые оценки неполны или имеют неверный формат.')
        return scores

    restored = up_load_artifact(path, read)
    return np.empty(0, dtype=np.float64) if restored is None else restored


def up_checkpoint_due(done, previous, total, started):
    return done == total or done - previous >= math.ceil(.05 * total) or time.monotonic() - started >= 900


def up_rng_state():
    name, words, offset, has_gauss, cached_gauss = np.random.get_state()
    return {
        'python': random.getstate(),
        'numpy': (name, torch.from_numpy(words.astype(np.int64)), offset, has_gauss, cached_gauss),
        'torch': torch.get_rng_state(),
        'cuda': torch.cuda.get_rng_state(up_DEVICE),
    }


def up_restore_rng(state):
    random.setstate(state['python'])
    name, words, offset, has_gauss, cached_gauss = state['numpy']
    np.random.set_state((name, words.numpy().astype(np.uint32), offset, has_gauss, cached_gauss))
    torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state(state['cuda'], up_DEVICE)


def up_save_training(path, signature, step, loss, weights, optimizer, scheduler):
    payload = {
        'signature': signature,
        'step': step,
        'loss': loss,
        'weights': weights,
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'rng': up_rng_state(),
    }
    up_atomic_artifact(path, lambda stream: torch.save(payload, stream))


def up_load_training(path, signature, total, expected_weights, optimizer, scheduler):
    def read(candidate):
        state = torch.load(candidate, map_location='cpu', weights_only=True)
        step = state['step']
        if state['signature'] != signature or not isinstance(step, int) or not 0 < step <= total:
            raise ValueError('Параметры сохранённого дообучения изменились.')
        if not math.isfinite(state['loss']) or state['scheduler']['last_epoch'] != step:
            raise ValueError('Некорректная позиция контрольной точки.')
        weights = state['weights']
        if set(weights) != set(expected_weights) or any(
            weights[name].shape != expected.shape or weights[name].dtype != expected.dtype
            for name, expected in expected_weights.items()
        ):
            raise ValueError('Некорректные веса контрольной точки.')
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        up_restore_rng(state['rng'])
        return state

    return up_load_artifact(path, read)


def up_remove_training_files(path):
    # The stage keeps two validated copies of its completed inference before
    # removing the larger optimizer checkpoints.
    for name in ('training.pt', 'training.pt.previous'):
        (path / name).unlink(missing_ok=True)


def up_persisted_prediction(batches, predict, path, signature, previous, total):
    values = np.empty(total, dtype=np.float64)
    values[:len(previous)] = previous
    done = checkpoint = len(previous)
    started = time.monotonic()
    for batch in batches:
        scores = np.asarray(predict(batch), dtype=np.float64)
        end = done + len(scores)
        values[done:end] = scores
        done = end
        if up_checkpoint_due(done, checkpoint, total, started):
            up_save_scores(path, signature, values[:done])
            checkpoint, started = done, time.monotonic()
    if done != total:
        raise RuntimeError(f'Оценки {path.name} неполны: {done} из {total}.')
    return values


In [ ]:
class up_PairDataset(Dataset):
    def __init__(self, d, with_labels=False):
        self.q = d.clean_query.tolist(); self.t = d.pair_text.tolist()
        self.with_labels = with_labels
        if with_labels:
            self.labels = d.relevance.to_numpy(dtype=np.float32)
            sizes = d.groupby('query_id').query_id.transform('size').to_numpy()
            self.weights = (len(d) / (d.query_id.nunique() * sizes)).astype(np.float32)
            assert np.isclose(self.weights.sum(), len(d), rtol=1e-5)
    def __len__(self): return len(self.q)
    def __getitem__(self, i):
        result = {'query':self.q[i], 'passage':self.t[i]}
        if self.with_labels: result.update(label=self.labels[i], weight=self.weights[i])
        return result

up_BGE_TOKENIZER = None


def up_bge_tokenizer():
    global up_BGE_TOKENIZER
    if up_BGE_TOKENIZER is None:
        up_BGE_TOKENIZER = AutoTokenizer.from_pretrained(up_MODEL_ID, revision=up_MODEL_REVISION, trust_remote_code=False)
    return up_BGE_TOKENIZER


def up_collate_pairs(rows):
    batch = up_bge_tokenizer()([r['query'] for r in rows], [r['passage'] for r in rows],
                      padding=True, truncation=True, max_length=up_MAX_LENGTH, return_tensors='pt')
    if 'label' in rows[0]:
        batch['labels'] = torch.tensor([r['label'] for r in rows], dtype=torch.float32)
        batch['weights'] = torch.tensor([r['weight'] for r in rows], dtype=torch.float32)
    return batch

def up_new_model():
    up_seed_all(up_SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        up_MODEL_ID, revision=up_MODEL_REVISION, trust_remote_code=False, use_safetensors=True,
        torch_dtype=torch.float32, attn_implementation='sdpa')
    assert model.config.num_labels == 1, 'Ожидается один reranker logit на пару.'
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
    return model.to(up_DEVICE)

@torch.inference_mode()
def up_predict_ce(model, d, path, signature, previous):
    model.eval()
    loader = DataLoader(up_PairDataset(d.iloc[len(previous):]), batch_size=up_INFER_BATCH, shuffle=False,
                        num_workers=0, collate_fn=up_collate_pairs, pin_memory=True)

    def predict(batch):
        inputs = {k: v.to(up_DEVICE, non_blocking=True) for k, v in batch.items()}
        with torch.autocast('cuda', dtype=torch.bfloat16):
            return model(**inputs).logits.reshape(-1).float().cpu().numpy()

    return up_persisted_prediction(tqdm(loader, desc='BGE inference', leave=False), predict,
                                path, signature, previous, len(d))


def up_training_budget(rows, microbatch, accumulation, fraction):
    """Exact shuffled prefix length, stopping only after an optimizer window."""
    assert rows > 0 and microbatch > 0 and accumulation > 0 and 0 < fraction <= 1
    batches = math.ceil(rows / microbatch)
    steps = math.ceil(batches / accumulation)
    planned_steps = math.ceil(fraction * steps)
    return {'batches_per_epoch':batches, 'steps_per_epoch':steps,
            'planned_optimizer_steps':planned_steps,
            'planned_processed_rows':min(rows, planned_steps * microbatch * accumulation)}

def up_train_bge(d, path, signature):
    fraction = up_BGE_FRACTION
    dataset = up_PairDataset(d, with_labels=True)
    budget = up_training_budget(len(dataset), up_MICROBATCH, up_ACCUMULATION, fraction)
    model = up_new_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=up_LEARNING_RATE,
                                 weight_decay=up_WEIGHT_DECAY, foreach=False)
    loader = DataLoader(dataset, batch_size=up_MICROBATCH, shuffle=True,
                        generator=torch.Generator().manual_seed(up_SEED), num_workers=0,
                        pin_memory=True, collate_fn=up_collate_pairs)
    steps_per_epoch = budget['steps_per_epoch']
    schedule_steps = up_EPOCHS * steps_per_epoch
    scheduler = get_linear_schedule_with_warmup(optimizer,
        num_warmup_steps=int(up_WARMUP_RATIO*schedule_steps), num_training_steps=schedule_steps)
    total = budget['planned_optimizer_steps']
    restored = up_load_training(path, signature, total, model.state_dict(), optimizer, scheduler)
    steps, loss_sum = 0, 0.0
    if restored is not None:
        steps, loss_sum = restored['step'], restored['loss']
        model.load_state_dict(restored['weights'])
        del restored
        print(f'BGE · продолжение с шага {steps}/{total}.')
    if steps == total:
        return model
    skip_batches = steps * up_ACCUMULATION
    seen = skip_batches * up_MICROBATCH
    window = 0
    checkpoint, started = steps, time.monotonic()
    optimizer.zero_grad(set_to_none=True)
    model.train()
    # Use the original DataLoader and seed. Skipping already seen batches
    # replays tokenization without advancing the restored dropout RNG.
    for bi, batch in enumerate(tqdm(loader, desc='BGE BCE train', total=len(loader))):
        if bi < skip_batches:
            continue
        weights = batch.pop('weights').to(up_DEVICE)
        labels = batch.pop('labels').to(up_DEVICE)
        inputs = {k:v.to(up_DEVICE, non_blocking=True) for k,v in batch.items()}
        with torch.autocast('cuda', dtype=torch.bfloat16):
            scores = model(**inputs).logits.reshape(-1).float()
            individual = torch.nn.functional.binary_cross_entropy_with_logits(scores, labels, reduction='none')
            loss = (individual*weights).sum()
        assert torch.isfinite(loss).item(), 'Non-finite loss'
        loss.backward()
        count = len(weights)
        window += count
        seen += count
        loss_sum += float(loss.detach())
        if (bi+1)%up_ACCUMULATION == 0 or bi+1 == len(loader):
            for parameter in model.parameters():
                if parameter.grad is not None: parameter.grad.div_(window)
            norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            assert torch.isfinite(norm).item(), 'Non-finite gradients'
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            steps += 1
            window = 0
            if up_checkpoint_due(steps, checkpoint, total, started):
                up_save_training(path, signature, steps, loss_sum, model.state_dict(), optimizer, scheduler)
                checkpoint, started = steps, time.monotonic()
            if steps == total:
                break
    assert steps == total and seen == budget['planned_processed_rows']
    return model


In [ ]:
up_QWEN_INSTRUCTION = 'Given a web search query, retrieve relevant passages that answer the query'
up_QWEN_PREFIX = ('<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. '
               'Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
up_QWEN_SUFFIX = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
up_QWEN_SPECS = {
    '4b': dict(repo='Qwen/Qwen3-Reranker-4B', revision='22e683669bc0f0bd69640a1354a6d0aebcfeede5', infer_batch=4),
    '8b': dict(repo='Qwen/Qwen3-Reranker-8B', revision='77d193c791ed757ca307ee72715aa132723da912', infer_batch=1),
}
up_ATTENTION_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj']

def up_escape_qwen_field(text):
    # Prevent data fields from producing Qwen role/think delimiters or template field headers.
    return text.replace('<', '＜').replace('>', '＞')

def up_qwen_body(query, document):
    return (f'<Instruct>: {up_QWEN_INSTRUCTION}\n<Query>: {up_escape_qwen_field(query)}'
            f'\n<Document>: {up_escape_qwen_field(document)}')

def up_qwen_answer_ids(tok):
    ids = [tok.convert_tokens_to_ids(word) for word in ('no', 'yes')]
    assert all(isinstance(i, int) and i not in tok.all_special_ids for i in ids)
    assert ids[0] != ids[1]
    assert all(tok.encode(word, add_special_tokens=False) == [i] for word, i in zip(('no', 'yes'), ids))
    return ids

def up_encode_qwen_batch(tok, texts, prefix, suffix, max_length):
    body_budget = max_length - len(prefix) - len(suffix)
    assert body_budget > 0 and suffix and tok.padding_side == 'left'
    body = tok(texts, padding=False, truncation=True, max_length=body_budget,
               add_special_tokens=False, return_attention_mask=False)['input_ids']
    batch = tok.pad({'input_ids': [prefix + ids + suffix for ids in body]},
                    padding=True, return_attention_mask=True, return_tensors='pt')
    assert batch.input_ids.shape[1] <= max_length
    assert batch.attention_mask[:, -1].eq(1).all().item()
    assert batch.input_ids[:, -len(suffix):].tolist() == [suffix] * len(texts)
    return batch

def up_yes_minus_no(last_logits, no_id, yes_id):
    return last_logits[:, yes_id].float() - last_logits[:, no_id].float()

def up_qwen_logit(model, batch, no_id, yes_id):
    logits = model(**batch, use_cache=False, logits_to_keep=1).logits
    assert logits.ndim == 3 and logits.shape[:2] == (len(batch['input_ids']), 1)
    return up_yes_minus_no(logits[:, -1, :], no_id, yes_id)

@torch.inference_mode()
def up_predict_qwen(model, tok, frame, batch_size, path, signature, previous):
    model.eval()
    no_id, yes_id = up_qwen_answer_ids(tok)
    prefix = tok.encode(up_QWEN_PREFIX, add_special_tokens=False)
    suffix = tok.encode(up_QWEN_SUFFIX, add_special_tokens=False)
    indices = tqdm(range(len(previous), len(frame), batch_size), desc='Qwen inference', leave=False)

    def predict(start):
        part = frame.iloc[start:start + batch_size]
        texts = [up_qwen_body(query, doc) for query, doc in zip(part.clean_query, part.pair_text)]
        batch = up_encode_qwen_batch(tok, texts, prefix, suffix, up_MAX_LENGTH).to(up_DEVICE)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            return up_qwen_logit(model, batch, no_id, yes_id).cpu().numpy()

    return up_persisted_prediction(indices, predict, path, signature, previous, len(frame))


def up_release_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def up_load_qwen(model_key, gradient_checkpointing=False):
    import bitsandbytes as bnb
    from peft import prepare_model_for_kbit_training

    spec = up_QWEN_SPECS[model_key]
    config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = Qwen3ForCausalLM.from_pretrained(
        spec['repo'], revision=spec['revision'], trust_remote_code=False,
        use_safetensors=True, dtype=torch.bfloat16, attn_implementation='sdpa',
        quantization_config=config, device_map={'': torch.cuda.current_device()},
    )
    assert any(isinstance(layer, bnb.nn.Linear4bit) for layer in model.modules())
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=gradient_checkpointing,
        gradient_checkpointing_kwargs={'use_reentrant': False},
    )
    assert model.get_input_embeddings().weight.device.type == 'cuda'
    model.config.use_cache = False
    return model

def up_new_adapter(model_key):
    from peft import LoraConfig, get_peft_model

    up_seed_all(up_SEED)
    modules = up_ATTENTION_MODULES
    model = get_peft_model(
        up_load_qwen(model_key, gradient_checkpointing=True),
        LoraConfig(r=up_QLORA_RANK, lora_alpha=up_QLORA_ALPHA, lora_dropout=0.0,
                   bias='none', task_type='CAUSAL_LM', target_modules=modules),
    )
    trainable = [name for name, param in model.named_parameters() if param.requires_grad]
    assert trainable and all('lora_' in name for name in trainable)
    assert all(any(f'.{module}.' in name for name in trainable) for module in modules)
    return model

def up_adapter_optimizer(model):
    parameters = [param for param in model.parameters() if param.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr=up_QLORA_LR, weight_decay=0.0, foreach=False)
    return optimizer, parameters

def up_qwen_loss(model, batch, labels, weights, answer_ids):
    with torch.autocast('cuda', dtype=torch.bfloat16):
        logits = up_qwen_logit(model, batch, *answer_ids)
        losses = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction='none')
        return (losses * weights).sum()

up_QWEN_TOKENIZERS = {}
def up_tokenizer_for(model_key):
    if model_key not in up_QWEN_TOKENIZERS:
        spec = up_QWEN_SPECS[model_key]
        up_QWEN_TOKENIZERS[model_key] = AutoTokenizer.from_pretrained(
            spec['repo'], revision=spec['revision'], trust_remote_code=False,
            padding_side='left', truncation_side='right',
        )
    return up_QWEN_TOKENIZERS[model_key]

def up_qlora_train(fit, model_key, path, signature):
    qwen_tokenizer = up_tokenizer_for(model_key)
    labels = fit.relevance.to_numpy(dtype=np.float32)
    texts = [up_qwen_body(q, p) for q, p in zip(fit.clean_query, fit.pair_text)]
    answer_ids = up_qwen_answer_ids(qwen_tokenizer)
    assert answer_ids == [2152, 9693]
    prefix = qwen_tokenizer.encode(up_QWEN_PREFIX, add_special_tokens=False)
    suffix = qwen_tokenizer.encode(up_QWEN_SUFFIX, add_special_tokens=False)
    steps_per_epoch = math.ceil(len(fit) / up_QLORA_ACCUMULATION)
    horizon = math.ceil(.5 * steps_per_epoch)
    order = torch.randperm(len(fit), generator=torch.Generator().manual_seed(up_SEED)).tolist()
    model = up_new_adapter(model_key)
    optimizer, parameters = up_adapter_optimizer(model)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(.05 * horizon), num_training_steps=horizon,
    )
    weights = {name: parameter for name, parameter in model.named_parameters() if parameter.requires_grad}
    restored = up_load_training(path, signature, horizon, weights, optimizer, scheduler)
    steps, loss_sum = 0, 0.0
    if restored is not None:
        steps, loss_sum = restored['step'], restored['loss']
        with torch.no_grad():
            for name, parameter in weights.items():
                parameter.copy_(restored['weights'][name])
        del restored
        print(f'Qwen {model_key} · продолжение с шага {steps}/{horizon}.')
    checkpoint, started = steps, time.monotonic()
    model.train()
    optimizer.zero_grad(set_to_none=True)
    for step in tqdm(range(steps + 1, horizon + 1), initial=steps, total=horizon, desc=f'Qwen {model_key} QLoRA'):
        window = order[(step - 1) * up_QLORA_ACCUMULATION:step * up_QLORA_ACCUMULATION]
        for i in window:
            batch = up_encode_qwen_batch(qwen_tokenizer, [texts[i]], prefix, suffix, up_MAX_LENGTH).to(up_DEVICE)
            y = torch.tensor([labels[i]], device=up_DEVICE)
            w = torch.ones(1, device=up_DEVICE, dtype=torch.float32)
            loss = up_qwen_loss(model, batch, y, w, answer_ids)
            (loss / len(window)).backward()
            loss_sum += float(loss.detach())
        norm = torch.nn.utils.clip_grad_norm_(parameters, 1.0)
        if not torch.isfinite(norm).item() or not math.isfinite(loss_sum):
            raise RuntimeError(f'Неконечный loss или градиент в Qwen {model_key}, шаг {step}.')
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        if up_checkpoint_due(step, checkpoint, horizon, started):
            up_save_training(path, signature, step, loss_sum,
                          {name: parameter.detach() for name, parameter in weights.items()}, optimizer, scheduler)
            checkpoint, started = step, time.monotonic()
    return model


In [ ]:
def up_inputs_fingerprint(frames):
    digest = hashlib.sha256()
    for frame in frames:
        header = [frame.columns.tolist(), len(frame)]
        digest.update(json.dumps(header, ensure_ascii=False, separators=(',', ':')).encode('utf-8') + b'\n')
        for record in frame.itertuples(index=False, name=None):
            line = json.dumps(record, ensure_ascii=False, separators=(',', ':'))
            digest.update(line.encode('utf-8') + b'\n')
    return digest.hexdigest()


def up_config_fingerprint(input_fingerprint):
    packages = {name: package_version(name) for name in (
        'numpy', 'pandas', 'scikit-learn', 'scipy', 'lightgbm', 'torch', 'transformers',
        'tokenizers', 'bitsandbytes', 'peft', 'accelerate',
    )}
    return up_fingerprint(up_CHECKPOINT_FORMAT, input_fingerprint, packages, torch.version.cuda,
                       torch.cuda.get_device_name(), torch.cuda.get_device_capability(),
                       up_MODEL_ID, up_MODEL_REVISION, up_QWEN_SPECS, up_SEED, up_MAX_LENGTH, up_MICROBATCH,
                       up_ACCUMULATION, up_INFER_BATCH, up_EPOCHS, up_BGE_FRACTION, up_LGB_ITERATIONS, up_LGB_PARAMS,
                       up_LEARNING_RATE, up_WEIGHT_DECAY, up_WARMUP_RATIO, up_THREADS,
                       up_N_FOLDS, up_QLORA_MICROBATCH, up_QLORA_ACCUMULATION, up_QLORA_LR,
                       up_QLORA_RANK, up_QLORA_ALPHA, up_QWEN_INSTRUCTION, up_QWEN_PREFIX, up_QWEN_SUFFIX)


def up_stage_scores(kind, fit, target, stage_dir, config_hash):
    folder = stage_dir / kind
    folder.mkdir(parents=True, exist_ok=True)
    cache = folder / 'scores.npz'
    training = folder / 'training.pt'
    fit_signature = up_fingerprint(config_hash, kind, up_ordered_id_hash(fit))
    signature = up_fingerprint(fit_signature, up_ordered_id_hash(target))
    batch = up_INFER_BATCH if kind == 'bge' else 1
    if kind in up_QWEN_SPECS:
        batch = up_QWEN_SPECS[kind]['infer_batch']
    previous = up_load_scores(cache, signature, len(target), batch)
    backup = cache.with_name('scores_complete.npz')
    if len(previous) != len(target):
        mirror = up_load_scores(backup, signature, len(target), batch)
        if len(mirror) == len(target):
            previous = mirror
            up_save_scores(cache, signature, previous)
    if len(previous) != len(target):
        if kind == 'lexical':
            features, model = up_fit_lexical(fit)
            previous = model.predict(features.transform(target), num_iteration=up_LGB_ITERATIONS)
            up_save_scores(cache, signature, previous)
        elif kind == 'bge':
            model = up_train_bge(fit, training, fit_signature)
            previous = up_predict_ce(model, target, cache, signature, previous)
            del model
            up_release_gpu()
        else:
            from peft import PeftModel

            adapter_dir = folder / 'adapter'
            model = up_qlora_train(fit, kind, training, fit_signature)
            model.save_pretrained(adapter_dir, safe_serialization=True, save_embedding_layers=False)
            del model
            up_release_gpu()
            # The v7.1 held and final runs loaded a fresh quantized base and the saved adapter.
            model = PeftModel.from_pretrained(up_load_qwen(kind), adapter_dir, is_trainable=False).eval()
            previous = up_predict_qwen(model, up_tokenizer_for(kind), target, batch, cache, signature, previous)
            del model
            up_release_gpu()
    mirror = up_load_scores(backup, signature, len(target), batch)
    if not np.array_equal(mirror, previous):
        up_save_scores(backup, signature, previous)
    up_remove_training_files(folder)
    return previous


def up_stage_table(fit, target, directory, config_hash):
    table = target[['id', 'query_id', 'document_id', 'article_id']].copy()
    for name, kind in zip(up_SCORE_COLUMNS, ('bge', 'lexical', '4b', '8b')):
        print(f'{directory.name} · {kind}: обучение {len(fit)}, инференс {len(target)} строк.')
        table[name] = up_stage_scores(kind, fit, target, directory, config_hash)
    return table


def up_fold_parts(train, geometry, outer):
    inner = (outer + 1) % up_N_FOLDS
    fit = train.loc[~geometry.fold.isin([outer, inner])].reset_index(drop=True)
    held = train.loc[geometry.fold.eq(outer)].reset_index(drop=True)
    assert set(fit.query_id).isdisjoint(held.query_id) and set(fit.document_id).isdisjoint(held.document_id)
    return fit, held


def run_release_upstream(data_dir, work_dir, *, inputs=None):
    """Train on component folds and return OOF and test scores in CSV row order."""
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError('Для BGE и Qwen требуется CUDA GPU с поддержкой BF16.')
    if 'logits_to_keep' not in inspect.signature(Qwen3ForCausalLM.forward).parameters:
        raise RuntimeError('Установите версии библиотек из requirements.txt.')
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    if inputs is None:
        inputs = upstream_read_inputs(data_dir)
    train, test, _ = inputs
    geometry = up_build_geometry(train)
    config_hash = up_config_fingerprint(up_inputs_fingerprint(inputs))
    train, test = up_prepare(train), up_prepare(test)
    torch.set_num_threads(up_THREADS)
    up_seed_all(up_SEED)
    parts = []
    for outer in range(up_N_FOLDS):
        fit, held = up_fold_parts(train, geometry, outer)
        part = up_stage_table(fit, held, work_dir / f'fold_{outer}', config_hash)
        part['relevance'] = held.relevance.to_numpy()
        part['fold'] = outer
        part['outer_fold'] = outer
        part['component'] = part.id.map(geometry.set_index('id').component)
        parts.append(part)
    oof = pd.concat(parts, ignore_index=True).set_index('id').loc[train.id].reset_index()
    test_scores = up_stage_table(train, test, work_dir / 'final', config_hash)
    assert oof.id.tolist() == train.id.tolist() and test_scores.id.tolist() == test.id.tolist()
    assert oof.component.notna().all()
    return oof, test_scores


## Линейное ранжирование, YetiRank и перестановка первых двух фрагментов

Три базовые оценки входят в линейную модель и YetiRank. Парный компаратор дополнительно учитывает оценку Qwen 4B и совпадение статьи. Порог обучается внутри каждого из пяти исходных фолдов, перестановку определяют голоса большинства.

In [ ]:
# Конфигурация и функции финальных голов из v17, v28 и v35.
import hashlib
import json
from itertools import combinations
from pathlib import Path
from time import monotonic
from uuid import uuid4

import catboost
import numpy as np
import pandas as pd
import sklearn
import scipy
from catboost import CatBoostRanker, Pool
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

REL35_RECIPE = "amml-meta-v2"
_rel35_SCORE_COLUMNS = ["bge", "lexical", "raw_qwen_8b_0.50"]
_rel35_PAIR_SCORES = _rel35_SCORE_COLUMNS + ["raw_attention_0.50"]
_rel35_PAIR_COLUMNS = _rel35_PAIR_SCORES + [f"z_{name}" for name in _rel35_PAIR_SCORES]
_rel35_ANCHOR_FEATURES = _rel35_SCORE_COLUMNS + [f"z_{name}" for name in _rel35_SCORE_COLUMNS]
_rel35_REGULARIZATION_C = 0.25
_rel35_SAMPLER_WEIGHTS = [0.495, 0.055, 0.45]
_rel35_EPSILON = 1e-12
_rel35_PARAMS = dict(loss_function="YetiRank", iterations=60, depth=3, learning_rate=0.08,
                     random_seed=2026, task_type="CPU", thread_count=2, verbose=False,
                     allow_writing_files=False)
_rel35_LINEAR_PARAMS = dict(solver="lbfgs", C=1.0, penalty="l2", fit_intercept=False,
                            max_iter=200, tol=1e-7)
_rel35_THRESHOLDS = (0.5, 0.525, 0.55, 0.575, 0.60, 0.65, 0.75, 1.0)
_rel35_INNER_FOLDS = 3


def _rel35_require(condition, message):
    if not condition:
        raise ValueError(message)


def _rel35_check_time(deadline):
    if monotonic() >= deadline:
        raise TimeoutError("Достигнут лимит времени на этапе построения рангов v35.")


def _rel35_zscore(series):
    values = series.to_numpy(dtype=np.float64)
    mean = values.mean()
    sd = values.std(ddof=0)
    return (values - mean) / (sd if sd > 1e-12 else 1.0)


def _rel35_zcols(frame, columns):
    grouped = frame.groupby("query_id", sort=False, observed=True)
    return {col: grouped[col].transform(_rel35_zscore) for col in columns}


def _rel35_ndcg_by_query(frame, score):
    _rel35_ordered = frame.sort_values(["query_id", score, "id"], ascending=[True, False, True])
    ideal = frame.sort_values(["query_id", "relevance"], ascending=[True, False])

    def dcg(data):
        rank = data.groupby("query_id", sort=False).cumcount() + 1
        gain = (np.exp2(data.relevance) - 1) / np.log2(rank + 1)
        return gain.where(rank <= 10, 0).groupby(data.query_id).sum()

    idcg = dcg(ideal)
    return dcg(_rel35_ordered).div(idcg.where(idcg > 0)).fillna(0)


def _rel35_add_z(frame):
    for name, values in _rel35_zcols(frame, _rel35_PAIR_SCORES).items():
        frame[f"z_{name}"] = values
    _rel35_require(np.isfinite(frame[_rel35_PAIR_COLUMNS].to_numpy()).all(), "Неконечные оценки.")


def _rel35_ordered(frame, score):
    return frame.sort_values(["query_id", score, "id"], ascending=[True, False, True])


def _rel35_pair_features(frame, left, right):
    values = frame[_rel35_PAIR_COLUMNS].to_numpy(dtype=float)
    articles = frame.article_id.to_numpy()
    difference = values[right] - values[left]
    same_article = (articles[right] == articles[left]) & (articles[left] != "")
    return np.column_stack([difference, difference[:, [4, 7]] * same_article[:, None]])


def _rel35_fit_pairs(frame):
    selected = _rel35_ordered(frame.assign(sampler=frame[_rel35_ANCHOR_FEATURES[3:]].to_numpy() @ _rel35_SAMPLER_WEIGHTS),
                       "sampler")
    targets = frame.relevance.to_numpy()
    first, second, weights = [], [], []
    fit_queries = 0
    top = selected.groupby("query_id", sort=False).head(4)
    for _, group in top.groupby("query_id", sort=False):
        candidates = group.index.to_numpy()
        mixed = [(i, j) for i, j in combinations(candidates, 2) if targets[i] != targets[j]]
        if mixed:
            fit_queries += 1
            for i, j in mixed:
                first.append(i)
                second.append(j)
                weights.append(1.0 / (2 * len(mixed)))
    _rel35_require(fit_queries > 0 and np.isclose(2 * sum(weights), fit_queries), "Некорректные веса пар.")
    features = _rel35_pair_features(frame, first, second)
    labels = targets[second]
    model = make_pipeline(
        StandardScaler(with_mean=False),
        LogisticRegression(C=_rel35_REGULARIZATION_C, solver="lbfgs", fit_intercept=False, max_iter=500),
    )
    model.fit(np.vstack([features, -features]), np.concatenate([labels, 1 - labels]),
              logisticregression__sample_weight=np.tile(weights, 2))
    return model, fit_queries, 2 * len(labels)


def _rel35_top_pairs(frame, score):
    ranking = _rel35_ordered(frame, score)
    positions = ranking.groupby("query_id", sort=False).cumcount()
    top = ranking.loc[positions.eq(0)]
    next_best = ranking.loc[positions.eq(1)]
    top_indices = pd.Series(top.index, index=top.query_id)
    first = top_indices.loc[next_best.query_id].to_numpy()
    second = next_best.index.to_numpy()
    return first, second, _rel35_pair_features(frame, first, second)


def _rel35_baseline_scores(fit, held):
    train_pool = Pool(fit[_rel35_ANCHOR_FEATURES], label=fit.relevance, group_id=fit.query_id)
    held_pool = Pool(held[_rel35_ANCHOR_FEATURES], group_id=held.query_id)
    model = CatBoostRanker(**_rel35_PARAMS).fit(train_pool)
    values = pd.Series(model.predict(held_pool, thread_count=_rel35_PARAMS["thread_count"]), index=held.index)
    tree = values.groupby(held.query_id, sort=False).transform(_rel35_zscore)
    return 0.8 * _rel35_zcols(held, ["v17_score"])["v17_score"] + 0.2 * tree


def _rel35_swap_scores(frame, baseline_column, left, right, swap):
    baseline = frame[baseline_column].to_numpy()
    output = baseline.copy()
    left, right = left[swap], right[swap]
    output[left], output[right] = baseline[right], baseline[left]
    # Ties at rank two need strict descending values to encode the requested order.
    # Reassign ordinal scores within those tied queries and keep their lower ranks.
    ties = frame.duplicated(["query_id", baseline_column], keep=False).to_numpy()
    tied_mask = ties[right]
    if tied_mask.any():
        tied_queries = frame.query_id.iloc[left[tied_mask]]
        ranking = _rel35_ordered(frame.loc[frame.query_id.isin(tied_queries)], baseline_column)
        ordinal = -ranking.groupby("query_id", sort=False).cumcount()
        output[ranking.index] = ordinal
        output[left[tied_mask]], output[right[tied_mask]] = -1, 0
    return output


def _rel35_pair_data(frame, inputs):
    blocks, weights = [], []
    skipped = 0
    labels = frame.relevance.to_numpy()
    for rows in frame.groupby("query_id", sort=False).indices.values():
        pos, neg = rows[labels[rows] == 1], rows[labels[rows] == 0]
        if not len(pos) or not len(neg):
            skipped += 1
            continue
        differences = (inputs[pos, None] - inputs[None, neg]).reshape(-1, inputs.shape[1])
        blocks.append(differences)
        weights.append(np.full(len(differences), 1 / (2 * len(differences))))
    _rel35_require(bool(blocks), "Нет пар с разными метками")
    forward = np.concatenate(blocks)
    weight = np.concatenate(weights)
    return (np.concatenate([forward, -forward]),
            np.repeat([1, 0], len(forward)),
            np.concatenate([weight, weight]), skipped)


def _rel35_six_inputs(frame):
    raw = frame[_rel35_SCORE_COLUMNS].to_numpy(dtype=float)
    z = frame[_rel35_ANCHOR_FEATURES[len(_rel35_SCORE_COLUMNS):]].to_numpy(dtype=float)
    return raw, z


def _rel35_fit_linear(frame, held):
    import warnings
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from threadpoolctl import threadpool_limits

    fit_rows = frame.sort_values(["query_id", "id"])
    _rel35_require(fit_rows.id.is_unique and fit_rows.relevance.isin([0, 1]).all(), "Некорректные fit ID/метки")
    train_raw, train_z = _rel35_six_inputs(fit_rows)
    raw, z = _rel35_six_inputs(held)
    scaler = StandardScaler().fit(train_raw)
    differences, labels, weights, skipped = _rel35_pair_data(
        fit_rows, np.column_stack([scaler.transform(train_raw), train_z]))
    with threadpool_limits(limits=4), warnings.catch_warnings(record=True) as messages:
        warnings.simplefilter("always")
        model = LogisticRegression(**_rel35_LINEAR_PARAMS).fit(differences, labels, sample_weight=weights)
    metadata = {
        "coefficient": model.coef_[0].tolist(),
        "raw_mean": scaler.mean_.tolist(), "raw_scale": scaler.scale_.tolist(),
        "directed_pairs": len(labels), "skipped_uniform_queries": skipped,
        "n_iter": int(model.n_iter_[0]),
        "converged": not any(issubclass(m.category, ConvergenceWarning) for m in messages),
        "warnings": [{"category": m.category.__name__, "message": str(m.message)} for m in messages],
    }
    prediction = model.decision_function(np.column_stack([scaler.transform(raw), z]))
    return prediction, metadata


def _rel35_logit_cutoff(probability):
    return np.inf if probability == 1.0 else np.log(probability / (1 - probability))


def _rel35_tune_cutoff(frame, deadline):
    held_parts = []
    for train_rows, held_rows in GroupKFold(n_splits=_rel35_INNER_FOLDS).split(frame, groups=frame.component):
        _rel35_check_time(deadline)
        fit = frame.iloc[train_rows].reset_index(drop=True)
        held = frame.iloc[held_rows].reset_index(drop=True)
        _rel35_require(set(fit.component).isdisjoint(held.component), "Пересечение компонентов inner.")
        held["v17_score"], linear_state = _rel35_fit_linear(fit, held)
        _rel35_require(linear_state["converged"], "Линейная v17 не сошлась на inner.")
        held["v28"] = _rel35_baseline_scores(fit, held)
        model, _, _ = _rel35_fit_pairs(fit)
        first, second, features = _rel35_top_pairs(held, "v28")
        logits = model.decision_function(features)
        held["flipped"] = _rel35_swap_scores(held, "v28", first, second, np.ones(len(logits), dtype=bool))
        gain = _rel35_ndcg_by_query(held, "flipped") - _rel35_ndcg_by_query(held, "v28")
        decisions = pd.DataFrame(dict(logit=logits, query_id=held.query_id.iloc[first].to_numpy()))
        part = gain.rename("gain").to_frame().join(decisions.set_index("query_id"))
        part["logit"] = part.logit.fillna(-np.inf)
        held_parts.append(part)
    validation = pd.concat(held_parts)
    _rel35_require(len(validation) == frame.query_id.nunique() and validation.index.is_unique,
            "Неполное покрытие inner.")
    choices = []
    for probability in _rel35_THRESHOLDS:
        swap = validation.logit.ge(_rel35_logit_cutoff(probability))
        choices.append((float(validation.gain.where(swap, 0).mean()), int(swap.sum()), probability))
    best_gain = max(choice[0] for choice in choices)
    tied = (choice for choice in choices if best_gain - choice[0] <= _rel35_EPSILON)
    gain, swaps, cutoff = min(tied, key=lambda choice: (choice[1], -choice[2]))
    return cutoff, gain, swaps


def _rel35_collapse_heads(test, decisions, fits):
    coefficient = np.array([m["coefficient"] for m in fits])
    scale = np.array([m["raw_scale"] for m in fits])
    location = np.array([m["raw_mean"] for m in fits])
    raw_weights = coefficient[:, :3] / scale
    effective = np.concatenate([raw_weights.mean(axis=0), coefficient[:, 3:].mean(axis=0)])
    intercept = float(-(raw_weights * location).sum(axis=1).mean())
    raw, z = _rel35_six_inputs(test)
    canonical = np.column_stack([raw, z]) @ effective
    centered = np.mean(decisions, axis=0) - intercept
    _rel35_require(np.allclose(canonical, centered, rtol=1e-12, atol=1e-12), "Не совпала линейная свёртка")
    # Допускаем перестановку лишь в численно неразличимых оценках.
    checks = (test[["query_id", "id"]].assign(canonical=canonical, centered=centered)
               .sort_values(["query_id", "canonical", "id"], ascending=[True, False, True]))
    jumps = checks.groupby("query_id").centered.diff().fillna(0)
    _rel35_require((jumps <= 1e-11).all(), "Линейная свёртка изменила порядок")
    return canonical, {"effective_coefficients": effective.tolist(), "dropped_intercept": intercept}


In [ ]:
def _rel35_inputs(upstream_train, upstream_test):
    identifiers = ["id", "query_id", "article_id", "document_id"]
    columns = identifiers + _rel35_PAIR_SCORES
    train = upstream_train[columns + ["relevance", "fold", "component"]].reset_index(drop=True).copy()
    test = upstream_test[columns].reset_index(drop=True).copy()
    for table, name in ((train, "train"), (test, "test")):
        _rel35_require(not table.empty and table.id.is_unique and table[identifiers].notna().all().all()
                       and table.id.ne("").all() and table.query_id.ne("").all(),
                       f"Пустые данные или некорректные ID {name}.")
        _rel35_require(np.isfinite(table[_rel35_PAIR_SCORES].to_numpy(dtype=np.float64)).all(),
                       f"Неконечные исходные оценки {name}.")
    _rel35_require(train.relevance.isin([0, 1]).all() and set(train.fold) == set(range(5)),
                   "Некорректные исходные фолды или метки v35.")
    _rel35_require(train.component.notna().all() and train.component.ne("").all(),
                   "Пустой идентификатор компонента v35.")
    _rel35_require(set(train.document_id).isdisjoint(test.document_id),
                   "Документы train и test пересекаются.")
    for col in ("query_id", "document_id", "component"):
        valid = train.groupby(col, sort=False)[["fold", "component"]].nunique().eq(1).all().all()
        _rel35_require(valid, f"{col} пересекает исходные фолды или компоненты.")
    return train, test


def _rel35_rank(frame, score):
    order = _rel35_ordered(frame, score)
    return (order.groupby("query_id", sort=False).cumcount() + 1).reindex(frame.index)


def _rel35_hash(data):
    return hashlib.sha256(data).hexdigest()


def _rel35_json(data):
    return json.dumps(data, ensure_ascii=False, allow_nan=False, sort_keys=True, separators=(",", ":")).encode()


def _rel35_cache_path(work_dir, train, test):
    config = [REL35_RECIPE, catboost.__version__, sklearn.__version__, scipy.__version__,
              np.__version__, pd.__version__, _rel35_PARAMS, _rel35_LINEAR_PARAMS,
              _rel35_PAIR_SCORES, _rel35_REGULARIZATION_C, _rel35_SAMPLER_WEIGHTS, _rel35_EPSILON,
              _rel35_THRESHOLDS, _rel35_INNER_FOLDS]
    digest = hashlib.sha256(_rel35_json(config))
    for frame in (train, test):
        fields = frame.columns.difference(_rel35_PAIR_SCORES, sort=False)
        digest.update(_rel35_json(frame[fields].values.tolist()))
        digest.update(frame[_rel35_PAIR_SCORES].to_numpy(dtype="<f8").tobytes(order="C"))
    return Path(work_dir) / f"v35-{digest.hexdigest()[:24]}.json"


def _rel35_cache_read(path, test):
    if path is None or not path.is_file():
        return None
    try:
        saved = json.loads(path.read_bytes())
        payload = saved["payload"]
        _rel35_require(saved["sha256"] == _rel35_hash(_rel35_json(payload)), "Повреждён кэш v35.")
        ranks = np.asarray(payload["ranks"])
        _rel35_require(ranks.shape == (len(test),) and ranks.dtype.kind in "iu",
                       "Кэш v35 содержит некорректные ранги.")
        result = test[["id", "query_id"]].assign(v35=ranks)
        ordered = result.sort_values(["query_id", "v35"])
        _rel35_require(ordered.v35.eq(ordered.groupby("query_id", sort=False).cumcount() + 1).all(),
                       "Кэш v35 содержит пропущенные или кратные позиции.")
        report = payload["report"]
        return result, {**report, "restored": True}
    except (OSError, ValueError, TypeError, KeyError):
        return None


def _rel35_cache_write(path, result, report):
    if path is None:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {"ranks": result.v35.tolist(), "report": report}
    temporary = path.with_name(f".{path.name}.{uuid4().hex}.tmp")
    try:
        temporary.write_bytes(_rel35_json({"payload": payload, "sha256": _rel35_hash(_rel35_json(payload))}))
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


def _rel35_fit_v28(train, test, deadline):
    # В v17 z-признаки пересчитывались после сортировки fit и в исходном порядке test.
    fit_data = train.sort_values(["query_id", "id"]).reset_index(drop=True)
    _rel35_add_z(fit_data)
    _rel35_add_z(test)
    linear_heads, linear_metadata = [], []
    for _, fold in fit_data.groupby("fold", sort=True):
        _rel35_check_time(deadline)
        prediction, metadata = _rel35_fit_linear(fold, test)
        _rel35_require(metadata["converged"], "Линейная голова v17 не сошлась.")
        linear_heads.append(prediction)
        linear_metadata.append(metadata)
    v17, _ = _rel35_collapse_heads(test, linear_heads, linear_metadata)

    # В v28 test сортировался перед вычислением z-признаков и запуском CatBoost.
    ordered_test = test.assign(v17=v17).sort_values(["query_id", "id"]).reset_index(drop=True)
    _rel35_add_z(ordered_test)
    predict_pool = Pool(ordered_test[_rel35_ANCHOR_FEATURES], group_id=ordered_test.query_id)
    tree_heads = []
    for _, fold in fit_data.groupby("fold", sort=True):
        _rel35_check_time(deadline)
        fit_pool = Pool(fold[_rel35_ANCHOR_FEATURES], label=fold.relevance, group_id=fold.query_id)
        model = CatBoostRanker(**_rel35_PARAMS).fit(fit_pool)
        prediction = pd.Series(model.predict(predict_pool), index=ordered_test.index)
        standardized = prediction.groupby(ordered_test.query_id, sort=False).transform(_rel35_zscore)
        tree_heads.append(standardized.to_numpy())
    baseline = ordered_test.groupby("query_id", sort=False).v17.transform(_rel35_zscore)
    v28 = 0.8 * baseline + 0.2 * np.mean(tree_heads, axis=0)
    test["v28"] = test.id.map(pd.Series(v28.to_numpy(), index=ordered_test.id))
    return fit_data


def release_v35(upstream_train, upstream_test, *, work_dir=None, deadline=float("inf")):
    """Train the final CPU heads and return test IDs, query IDs and 1-based ranks."""
    train, test = _rel35_inputs(upstream_train, upstream_test)
    cache = _rel35_cache_path(work_dir, train, test) if work_dir is not None else None
    restored = _rel35_cache_read(cache, test)
    if restored is not None:
        return restored

    fit_data = _rel35_fit_v28(train, test, deadline)
    _rel35_require(np.isfinite(test.v28).all(), "Неконечные оценки v28.")
    left, right, features = _rel35_top_pairs(test, "v28")

    cutoffs, votes = [], []
    for _, fold in fit_data.groupby("fold", sort=True):
        _rel35_check_time(deadline)
        fold = fold.reset_index(drop=True)
        cutoff, _, _ = _rel35_tune_cutoff(fold, deadline)
        model, _, _ = _rel35_fit_pairs(fold)
        cutoffs.append(cutoff)
        votes.append(model.decision_function(features) >= _rel35_logit_cutoff(cutoff))
    swap = np.sum(votes, axis=0) >= 3
    test["v35_score"] = _rel35_swap_scores(test, "v28", left, right, swap)
    test["v35"] = _rel35_rank(test, "v35_score")
    expected = _rel35_rank(test, "v28").to_numpy(copy=True)
    expected[left[swap]], expected[right[swap]] = expected[right[swap]], expected[left[swap]]
    _rel35_require(np.array_equal(test.v35, expected), "v35 изменила порядок за пределами выбранных пар.")
    result = test[["id", "query_id", "v35"]].copy()
    report = {
        "final_cutoffs": cutoffs,
        "test_swaps": int(swap.sum()),
        "catboost": catboost.__version__,
        "scikit_learn": sklearn.__version__,
        "restored": False,
    }
    _rel35_cache_write(cache, result, report)
    return result, report


## Jina v3.5 — модель и упаковка фрагментов

In [ ]:
import json
import math
import os
from pathlib import Path
import re
import subprocess
import sys
import time

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F


V43_MODEL_ID = 'jinaai/jina-reranker-v3.5'
V43_MODEL_REVISION = 'e8a93f33f0b22108f8c2364f8484ce3422552fbc'
V43_LORA_RANK = 8
V43_LORA_ALPHA = 16
_V43_DOC_MARKER = '<|embed_token|>'
_V43_QUERY_MARKER = '<|rerank_token|>'
_V43_MODEL_FILES = (
    'config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json',
    'added_tokens.json', 'special_tokens_map.json', 'merges.txt', 'vocab.json',
)
_V43_SYSTEM = (
    '<|im_start|>system\n'
    'You rank search passages by relevance. For questions, prefer passages that provide the requested answer.'
    '<|im_end|>\n<|im_start|>user\n'
)
_V43_SUFFIX = '\n</query><|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
_V43_STRUCTURAL = re.compile(r'<(?:\|[^<>\r\n]{1,80}\||/?(?:passage|query|instruct|think|tool_call|tool_response)(?:\s+[^<>]*)?)>', re.IGNORECASE)


def _v43_remaining(deadline):
    remaining = None if deadline is None else deadline - time.monotonic()
    if remaining is not None and remaining <= 0:
        raise TimeoutError('Истёк лимит времени v43.')
    return None if remaining == math.inf else remaining


def _v43_snapshot(deadline):
    from huggingface_hub import snapshot_download
    from huggingface_hub.errors import LocalEntryNotFoundError

    def complete(location):
        return location is not None and all((Path(location) / name).is_file() for name in _V43_MODEL_FILES)

    try:
        path = snapshot_download(V43_MODEL_ID, revision=V43_MODEL_REVISION, local_files_only=True)
    except (FileNotFoundError, LocalEntryNotFoundError):
        path = None
    if complete(path):
        return Path(path)
    program = (
        'import json, sys\n'
        'from huggingface_hub import snapshot_download\n'
        'repo, revision, *files = sys.argv[1:]\n'
        'print(json.dumps(snapshot_download(repo, revision=revision, allow_patterns=files)))\n'
    )
    print(f'[v43] Загрузка весов {V43_MODEL_ID}@{V43_MODEL_REVISION}.', flush=True)
    try:
        result = subprocess.run(
            [sys.executable, '-c', program, V43_MODEL_ID, V43_MODEL_REVISION, *_V43_MODEL_FILES],
            check=True, text=True, stdout=subprocess.PIPE, timeout=_v43_remaining(deadline),
            env={**os.environ, 'HF_HUB_DISABLE_XET': '1'},
        )
    except subprocess.TimeoutExpired as exc:
        raise TimeoutError('Не хватило времени для загрузки весов Jina.') from exc
    path = json.loads(result.stdout.splitlines()[-1])
    if not complete(path):
        raise RuntimeError('Снимок Jina неполный.')
    return Path(path)


class _V43LoRA(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.lora_a = nn.Parameter(torch.empty(V43_LORA_RANK, base.in_features, dtype=torch.float32))
        self.lora_b = nn.Parameter(torch.zeros(base.out_features, V43_LORA_RANK, dtype=torch.float32))
        nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))

    def forward(self, x):
        frozen = self.base(x)
        with torch.autocast(device_type=x.device.type, enabled=False):
            update = F.linear(F.linear(x.float(), self.lora_a), self.lora_b)
            update = update * (V43_LORA_ALPHA / V43_LORA_RANK)
        return frozen + update.to(frozen.dtype)


class _V43Jina(nn.Module):
    def __init__(self, backbone, projector):
        super().__init__()
        self.backbone = backbone
        self.projector = projector

    def forward(self, input_ids, marker_positions):
        hidden = self.backbone(input_ids=input_ids, use_cache=False, return_dict=True).last_hidden_state
        selected = hidden[0].index_select(0, marker_positions)
        with torch.autocast(device_type=selected.device.type, enabled=False):
            embeddings = self.projector(selected.float())
            return F.cosine_similarity(embeddings[:-1], embeddings[-1:], dim=-1)


def _v43_marker_ids(tokenizer):
    markers = tuple(tokenizer.convert_tokens_to_ids(token) for token in (_V43_DOC_MARKER, _V43_QUERY_MARKER))
    converted = tokenizer.convert_ids_to_tokens(list(markers)) if None not in markers else []
    if list(converted) != [_V43_DOC_MARKER, _V43_QUERY_MARKER] or markers[0] == markers[1]:
        raise RuntimeError('Токенизатор не содержит ожидаемые разделители Jina.')
    return markers


def v43_load_model(deadline, *, device='cuda', local_model=None):
    from safetensors import safe_open
    from transformers import AutoTokenizer, Qwen3Config, Qwen3Model

    device = torch.device(device)
    if device.type == 'cuda' and (not torch.cuda.is_available() or not torch.cuda.is_bf16_supported()):
        raise RuntimeError('Для обучения v43 требуется CUDA с BF16.')
    _v43_remaining(deadline)
    path = _v43_snapshot(deadline) if local_model is None else Path(local_model)
    tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=False, local_files_only=True, use_fast=True)
    marker_ids = _v43_marker_ids(tokenizer)
    config_data = json.loads((path / 'config.json').read_text(encoding='utf-8'))
    if config_data.get('model_type') != 'qwen3':
        raise RuntimeError('Ожидаются веса Jina с основой Qwen3.')
    config = Qwen3Config.from_dict(config_data)
    layer_types = config_data.get('layer_types', [])
    if (len(layer_types) != config.num_hidden_layers or layer_types != config.layer_types
            or layer_types[-1] != 'full_attention' or 'sliding_attention' not in layer_types
            or not config.sliding_window or max(marker_ids) >= config.vocab_size):
        raise RuntimeError('Конфигурация Qwen3 не сохранила гибридные слои Jina.')
    if local_model is None and (marker_ids != (151670, 151671) or config.sliding_window != 1024
                                or layer_types.count('sliding_attention') != 16
                                or layer_types.count('full_attention') != 12 or config.hidden_size != 1024):
        raise RuntimeError('Снимок Jina отличается от ожидаемой модели v3.5.')
    config._attn_implementation = 'sdpa'
    config.use_cache = False
    config.output_hidden_states = False
    config.output_attentions = False
    backbone = Qwen3Model(config)
    backbone_dtype = torch.bfloat16 if device.type == 'cuda' else torch.float32
    with safe_open(path / 'model.safetensors', framework='pt', device='cpu') as checkpoint:
        keys = set(checkpoint.keys())
        expected = {'model.' + key for key in backbone.state_dict()}
        projector_keys = {'projector.0.weight', 'projector.2.weight'}
        allowed_extra = {'lm_head.weight'}
        missing = (expected | projector_keys) - keys
        unexpected = keys - expected - projector_keys - allowed_extra
        if missing or unexpected:
            raise RuntimeError(f'Несовместимые ключи safetensors: missing={sorted(missing)}, unexpected={sorted(unexpected)}')
        weights = {}
        for key in sorted(expected):
            _v43_remaining(deadline)
            weights[key.removeprefix('model.')] = checkpoint.get_tensor(key).to(backbone_dtype)
        backbone.load_state_dict(weights, strict=True, assign=True)
        del weights
        first = checkpoint.get_tensor('projector.0.weight').float()
        last = checkpoint.get_tensor('projector.2.weight').float()
    half_width = config.hidden_size // 2
    expected_output = 512 if local_model is None else last.shape[0]
    projector = nn.Sequential(
        nn.Linear(config.hidden_size, half_width, bias=False), nn.ReLU(),
        nn.Linear(half_width, expected_output, bias=False),
    )
    projector.load_state_dict({'0.weight': first, '2.weight': last}, strict=True)
    backbone.requires_grad_(False)
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(43)
        for layer in backbone.layers:
            attention = layer.self_attn
            for name in ('q_proj', 'k_proj', 'v_proj', 'o_proj'):
                setattr(attention, name, _V43LoRA(getattr(attention, name)))
    backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model = _V43Jina(backbone, projector).to(device).eval()
    _v43_remaining(deadline)
    return tokenizer, model


def _v43_encode(tokenizer, texts):
    return tokenizer(texts, add_special_tokens=False, verbose=False)['input_ids']


def _v43_safe_text(text):
    return _V43_STRUCTURAL.sub('', text.replace('\x00', ' ')).strip()


def _v43_document_lengths(lengths, budget):
    if sum(lengths) <= budget:
        return lengths
    left, right = 1, max(lengths)
    while left < right:
        middle = (left + right + 1) // 2
        if sum(min(length, middle) for length in lengths) <= budget:
            left = middle
        else:
            right = middle - 1
    limits = [min(length, left) for length in lengths]
    remaining = budget - sum(limits)
    for index, length in enumerate(lengths):
        if remaining and length > limits[index]:
            limits[index] += 1
            remaining -= 1
    return limits


def v43_groups(tokenizer, frame, deadline, max_tokens=4096, max_doc_tokens=256):
    if max_tokens < 1 or max_doc_tokens < 1 or frame.id.duplicated().any():
        raise ValueError('Некорректный лимит токенов или повторяющиеся ID.')
    markers = _v43_marker_ids(tokenizer)
    doc_marker, query_marker = markers
    labelled = 'relevance' in frame.columns
    records = []
    working = frame.reset_index(drop=True)
    suffix = _v43_encode(tokenizer, _V43_SUFFIX)
    headers = {}
    for query_id, group in working.groupby('query_id', sort=False, dropna=False):
        _v43_remaining(deadline)
        query_values = group['query'].fillna('').astype(str).unique()
        if len(query_values) != 1 or not (query := _v43_safe_text(query_values[0])):
            raise ValueError(f'Некорректный текст запроса {query_id!r}.')
        count = len(group)
        if count not in headers:
            starts = ['\n<passage id="0">']
            starts.extend(f'\n</passage>\n<passage id="{index}">' for index in range(1, count))
            headers[count] = _v43_encode(tokenizer, starts)
        starts = headers[count]
        prefix, query_end = _v43_encode(tokenizer, [
            _V43_SYSTEM + f'I will provide you with {count} passages, each indicated by a numerical identifier. '
            f'Rank the passages based on their relevance to query: {query}',
            '\n</passage>\n<query>\n' + query,
        ])
        candidate_titles = group['title'].fillna('').astype(str)
        candidate_texts = group['text'].fillna('').astype(str)
        documents = []
        for title, passage in zip(candidate_titles, candidate_texts):
            title, passage = _v43_safe_text(title), _v43_safe_text(passage)
            documents.append('\n' + (title + '\n' if title else '') + passage)
        token_docs = _v43_encode(tokenizer, documents)
        budget = max_tokens - len(prefix) - len(query_end) - len(suffix) - sum(map(len, starts)) - count - 1
        if budget < count:
            raise ValueError(f'Запрос {query_id!r} и разделители не помещаются в {max_tokens} токенов.')
        sizes = _v43_document_lengths([min(max_doc_tokens, len(part)) for part in token_docs], budget)
        clipped_docs = [tuple(part[:size]) for part, size in zip(token_docs, sizes)]
        safe_sections = [prefix, query_end, suffix, *starts, *clipped_docs]
        if any(doc_marker in part or query_marker in part for part in safe_sections):
            raise ValueError(f'Лишний разделитель в запросе {query_id!r}.')
        labels = group['relevance'].to_numpy(dtype=np.float32, copy=True) if labelled else None
        if labels is not None and not np.isin(labels, [0, 1]).all():
            raise ValueError(f'Недопустимые метки запроса {query_id!r}.')
        records.append({
            'query_id': str(query_id), 'rows': group.index.to_numpy(dtype=np.int64, copy=True),
            'ids': group.id.astype(str).tolist(), 'labels': labels,
            'token_count': max_tokens - budget + sum(sizes),
            'truncated_docs': sum(size < len(part) for part, size in zip(token_docs, sizes)),
            'prefix': prefix, 'starts': starts, 'documents': clipped_docs, 'query_end': query_end,
            'suffix': suffix, 'marker_ids': markers,
        })
    return records


def v43_group_scores(model, record, order):
    order = [int(index) for index in order]
    if sorted(order) != list(range(len(record['ids']))):
        raise ValueError('Порядок входа должен переставлять все фрагменты запроса.')
    doc_marker, query_marker = record['marker_ids']
    packed = list(record['prefix'])
    positions = []
    for start, document_index in zip(record['starts'], order):
        packed.extend(start)
        packed.extend(record['documents'][document_index])
        positions.append(len(packed))
        packed.append(doc_marker)
    packed.extend(record['query_end'])
    positions.append(len(packed))
    packed.append(query_marker)
    packed.extend(record['suffix'])
    device = next(model.parameters()).device
    return model(
        torch.tensor([packed], dtype=torch.long, device=device),
        torch.tensor(positions, dtype=torch.long, device=device),
    )


def v43_trainable_state(model):
    return {name: parameter.detach().to(device='cpu', copy=True)
            for name, parameter in model.named_parameters() if parameter.requires_grad}


def v43_assert_finite_gradients(model):
    trainable = [(name, parameter) for name, parameter in model.named_parameters() if parameter.requires_grad]
    missing = [name for name, parameter in trainable if parameter.grad is None]
    if not trainable or missing:
        raise RuntimeError(f'Не получены градиенты параметров v43: {missing}')
    norms = torch.nn.utils.get_total_norm([parameter.grad for _, parameter in trainable], error_if_nonfinite=True)
    if not bool(norms > 0):
        raise RuntimeError('Все градиенты параметров v43 оказались нулевыми.')


def v43_restore_trainable(model, state):
    parameters = {name: parameter for name, parameter in model.named_parameters() if parameter.requires_grad}
    if set(parameters) != set(state):
        raise ValueError('Ключи адаптера v43 в контрольной точке отличаются от модели.')
    for name, parameter in parameters.items():
        saved = state[name]
        if (not isinstance(saved, torch.Tensor) or saved.shape != parameter.shape
                or saved.dtype != parameter.dtype or not torch.isfinite(saved).all()):
            raise ValueError(f'Некорректный тензор адаптера {name!r}.')
    with torch.no_grad():
        for name, parameter in parameters.items():
            parameter.copy_(state[name])


## Финальное дообучение Jina на всех запросах

In [ ]:
import hashlib
from zipfile import ZipFile

def _v43_digest(identity, groups):
    digest = hashlib.sha256(json.dumps(identity, sort_keys=True, default=str).encode())
    for group in groups:
        labels = group["labels"]
        values = [str(group["query_id"]), list(map(str, group["ids"])),
                  np.asarray(group["rows"]).tolist(),
                  None if labels is None else np.asarray(labels).tolist()]
        digest.update(json.dumps(values, ensure_ascii=False).encode())
    return digest.hexdigest()

def _v43_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("wb") as handle:
        torch.save(payload, handle)
        handle.flush()
        os.fsync(handle.fileno())
    previous = path.with_suffix(path.suffix + ".previous")
    if path.exists():
        os.replace(path, previous)
    os.replace(tmp, path)

def _v43_load(path, signature):
    path = Path(path)
    problems = []
    for candidate in (path, path.with_suffix(path.suffix + ".previous")):
        if not candidate.exists():
            continue
        try:
            with ZipFile(candidate) as archive:
                broken = archive.testzip()
                if broken is not None:
                    raise ValueError(f"Bad CRC at {broken}")
            payload = torch.load(candidate, map_location="cpu", weights_only=True)
        except Exception as exc:
            problems.append(f"{candidate}: {exc}")
            continue
        if payload["signature"] != signature:
            raise ValueError(f"Контрольная точка относится к другим входам или настройкам: {candidate}")
        if candidate != path:
            print(f"Recovered previous checkpoint {candidate}", flush=True)
            os.replace(candidate, path)
        return payload
    if problems:
        raise RuntimeError("No readable checkpoint. " + " / ".join(problems))
    return None

def _v43_order(group, seed):
    key = f'{seed}|{group["query_id"]}'.encode()
    seed = int.from_bytes(hashlib.sha256(key).digest()[:8], "little")
    return np.random.default_rng(seed).permutation(len(group["ids"]))

def _v43_features(scores, group, order):
    ids = np.asarray(group["ids"], dtype=str)[order]
    labels = np.asarray(group["labels"])[order]
    rank = np.lexsort((ids, -scores.detach().float().cpu().numpy()))
    return labels, rank

def _v43_discount(size):
    discount = np.zeros(size, np.float64)
    top = min(10, size)
    discount[:top] = 1.0 / np.log2(np.arange(top) + 2.0)
    return discount

def _v43_loss(scores, group, order):
    scores = scores.float() / 0.05
    labels, ranked = _v43_features(scores, group, order)
    gains = np.exp2(labels) - 1.0
    discounts = _v43_discount(len(labels))
    ideal = float(np.dot(np.sort(gains)[::-1], discounts))
    if ideal <= 0.0:
        return scores.sum() * 0.0
    rank_discount = np.empty_like(discounts)
    rank_discount[ranked] = discounts
    better, worse = np.nonzero(labels[:, None] > labels[None, :])
    weights = ((gains[better] - gains[worse])
               * np.abs(rank_discount[better] - rank_discount[worse]) / ideal)
    nonzero = weights > 0.0
    if nonzero.any():
        better = torch.as_tensor(better[nonzero], device=scores.device)
        worse = torch.as_tensor(worse[nonzero], device=scores.device)
        weight = torch.as_tensor(weights[nonzero], dtype=scores.dtype, device=scores.device)
        pair_loss = (weight * F.softplus(scores[worse] - scores[better])).sum() / weight.sum()
    else:
        pair_loss = scores.sum() * 0.0
    positive = torch.as_tensor(labels > 0, device=scores.device)
    list_loss = -F.log_softmax(scores, dim=0)[positive].mean()
    return 0.7 * pair_loss + 0.3 * list_loss

def _v43_checkpoint_rng():
    return dict(cpu=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])

def _v43_restore_rng(rng):
    torch.set_rng_state(rng["cpu"])
    if rng["cuda"]:
        torch.cuda.set_rng_state_all(rng["cuda"])

def _jina_fit(model, groups, directory, identity, deadline):
    # v44 selected one complete epoch from its two-epoch scheduler horizon.
    seed = 43
    batch_size = 4
    total = math.ceil(len(groups) / batch_size)
    horizon = 2 * total
    path = directory / 'fit.pt'
    signature = _v43_digest(['fit', identity, seed, 2, total, None], groups)

    parameters = [p for p in model.parameters() if p.requires_grad]
    if not parameters or any(p.dtype != torch.float32 for p in parameters):
        raise ValueError('Параметры LoRA и проектора Jina должны быть FP32.')
    torch.manual_seed(seed)
    optimizer = torch.optim.AdamW(parameters, lr=5e-5)
    warmup = max(1, math.ceil(0.05 * horizon))

    def lr_factor(step):
        if step < warmup:
            return (step + 1) / warmup
        return max(0.1, 1.0 - (step - warmup) / max(1, horizon - warmup))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    step = 0
    checkpoint = _v43_load(path, signature)
    if checkpoint is not None:
        step = checkpoint['step']
        if type(step) is not int or not 0 <= step <= total:
            raise ValueError(f'Некорректная позиция обучения Jina в {path}.')
        v43_restore_trainable(model, checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        scheduler.load_state_dict(checkpoint['scheduler'])
        _v43_restore_rng(checkpoint['rng'])
        del checkpoint

    def save():
        _v43_save(dict(signature=signature, step=step, model=v43_trainable_state(model),
                       optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                       rng=_v43_checkpoint_rng()), path)

    last_saved = step
    last_save_time = time.monotonic()
    interval = max(1, math.ceil(0.05 * total))
    order_of_groups = np.random.default_rng(np.random.SeedSequence([seed, 0])).permutation(len(groups))
    model.train()
    print(f'Jina: обучение с шага {step}/{total}', flush=True)
    while step < total:
        _v43_remaining(deadline)
        group_indices = order_of_groups[batch_size * step:batch_size * (step + 1)]
        optimizer.zero_grad(set_to_none=True)
        for index in group_indices:
            group = groups[int(index)]
            order = _v43_order(group, f'{seed}|train|0')
            loss = _v43_loss(v43_group_scores(model, group, order), group, order)
            if not torch.isfinite(loss):
                raise ValueError(f'Jina получила неконечную потерю для запроса {group["query_id"]}.')
            (loss / len(group_indices)).backward()
        torch.nn.utils.clip_grad_norm_(parameters, 1.0, error_if_nonfinite=True)
        optimizer.step()
        scheduler.step()
        step += 1
        now = time.monotonic()
        if (step == total or step - last_saved >= interval or now - last_save_time >= 900
                or now >= deadline):
            save()
            last_saved, last_save_time = step, now
            print(f'Jina: сохранён шаг обучения {step}/{total}', flush=True)
    model.zero_grad(set_to_none=True)


## Четыре порядка фрагментов на тесте

In [ ]:
def _v47_order(group):
    """Choose a second order whose reversal is also new for this query."""
    if len(group["ids"]) <= 2:
        return None
    baseline = _v43_order(group, "v43-eval")
    order = _v43_order(group, "v47-eval")
    if np.array_equal(order, baseline) or np.array_equal(order, baseline[::-1]):
        # An adjacent swap cannot reverse a list of at least three candidates.
        order[[0, 1]] = order[[1, 0]]
    return order

def _jina_predict(model, groups, length, directory, identity, deadline, *, baseline=None):
    if baseline is None:
        phase, inputs_sha = 'two', None
    else:
        phase, inputs_sha = 'four', hashlib.sha256(baseline.tobytes()).hexdigest()
        baseline = torch.from_numpy(baseline)
    path = directory / f'test_{phase}.pt'
    signature = _v43_digest(['predict', identity, length, phase, inputs_sha], groups)
    previous = _v43_load(path, signature)
    scores = torch.full((length,), torch.nan, dtype=torch.float32)
    cursor = 0
    if previous is not None:
        cursor, scores = previous['cursor'], previous['scores']
        if (type(cursor) is not int or not 0 <= cursor <= len(groups)
                or not isinstance(scores, torch.Tensor) or scores.dtype != torch.float32
                or scores.shape != (length,)):
            raise ValueError(f'Некорректная контрольная точка Jina в {path}.')
        completed = np.zeros(length, dtype=bool)
        for group in groups[:cursor]:
            completed[group['rows']] = True
        if not (torch.isfinite(scores[completed]).all() and torch.isnan(scores[~completed]).all()):
            raise ValueError(f'Позиция инференса Jina расходится с оценками в {path}.')
    last_saved = cursor
    last_save_time = time.monotonic()
    interval = max(1, math.ceil(0.05 * len(groups)))
    model.eval()
    pending_rows = None
    try:
        with torch.inference_mode():
            for index in range(cursor, len(groups)):
                _v43_remaining(deadline)
                group = groups[index]
                order = _v43_order(group, 'v43-eval') if baseline is None else _v47_order(group)
                if order is None:
                    rows = torch.as_tensor(group['rows'], dtype=torch.long)
                    values = baseline[rows]
                else:
                    rows = torch.as_tensor(group['rows'][order], dtype=torch.long)
                    forward = v43_group_scores(model, group, order).detach().float().cpu()
                    reverse = v43_group_scores(model, group, order[::-1].copy()).detach().float().cpu()
                    if forward.shape != rows.shape or reverse.shape != rows.shape:
                        raise ValueError(f'Jina вернула неверное число оценок для запроса {group["query_id"]}.')
                    values = 0.5 * (forward + reverse.flip(0))
                    if baseline is not None:
                        values = 0.5 * (baseline[rows] + values)
                if not torch.isfinite(values).all():
                    raise ValueError(f'Jina вернула неконечные оценки для запроса {group["query_id"]}.')
                pending_rows = rows
                scores[rows] = values
                cursor = index + 1
                pending_rows = None
                now = time.monotonic()
                if cursor - last_saved >= interval or now - last_save_time >= 900:
                    _v43_save(dict(signature=signature, cursor=cursor, scores=scores), path)
                    last_saved, last_save_time = cursor, now
                    print(f'Jina: проход {phase}, {cursor}/{len(groups)} запросов', flush=True)
    finally:
        if cursor != last_saved:
            if pending_rows is not None and cursor == index:
                scores[pending_rows] = torch.nan
            _v43_save(dict(signature=signature, cursor=cursor, scores=scores), path)
    return scores.numpy()


## Запуск и контрольные точки Jina

In [ ]:

from contextlib import contextmanager
from importlib.metadata import version


_JINA_CACHE_FORMAT = 'amml-jina-v2'
_JINA_MAX_TOKENS = 4096
_JINA_FIT_DOC_TOKENS = 256
_JINA_TEST_DOC_TOKENS = 512

def _jina_frame_hash(frame, training):
    fields = ['id', 'query_id', 'query', 'title', 'text']
    if training:
        fields.append('relevance')
    if frame.empty or not set(fields).issubset(frame.columns):
        raise ValueError('Jina ожидает непустые данные с ID, запросами и текстами фрагментов.')
    if (not frame.id.is_unique or frame[['id', 'query_id']].isna().any().any()
            or frame[['id', 'query_id']].eq('').any().any()):
        raise ValueError('Jina получила пустые идентификаторы или повторяющийся ID фрагмента.')
    if training and not frame.relevance.isin([0, 1]).all():
        raise ValueError('Jina ожидает числовые метки relevance 0 и 1.')
    digest = hashlib.sha256()
    for record in frame[fields].itertuples(index=False, name=None):
        line = json.dumps(record, ensure_ascii=False, separators=(',', ':'))
        digest.update(line.encode('utf-8') + b'\n')
    return digest.hexdigest()


def _jina_runtime():
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported(including_emulation=False):
        raise RuntimeError('Для Jina требуется NVIDIA GPU с аппаратной поддержкой BF16.')
    return {
        'device': torch.cuda.get_device_name(),
        'capability': torch.cuda.get_device_capability(),
        'torch_cuda': torch.version.cuda,
        'packages': {name: version(name) for name in ('torch', 'numpy', 'pandas', 'transformers',
                                                      'tokenizers', 'safetensors', 'huggingface-hub')},
        'sdpa': [torch.backends.cuda.flash_sdp_enabled(), torch.backends.cuda.mem_efficient_sdp_enabled(),
                 torch.backends.cuda.math_sdp_enabled(), torch.backends.cuda.cudnn_sdp_enabled()],
    }


@contextmanager
def _jina_run_lock(path):
    # Closing the file also releases this lock if the kernel stops.
    with path.open('a+b') as handle:
        if handle.tell() == 0:
            handle.write(b'0')
            handle.flush()
        handle.seek(0)
        try:
            if os.name == 'nt':
                import msvcrt
                msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl
                fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError as exc:
            raise RuntimeError(f'Jina уже запущена в другом ядре для {path.parent}.') from exc
        yield


def run_jina(train, test, run_dir, deadline):
    """Fine-tune on all training queries and return mean scores in test CSV row order."""
    if 'relevance' in test:
        test = test.drop(columns=['relevance'])
    train_identity = _jina_frame_hash(train, True)
    test_identity = _jina_frame_hash(test, False)
    settings = [_JINA_CACHE_FORMAT, V43_MODEL_ID, V43_MODEL_REVISION, V43_LORA_RANK, V43_LORA_ALPHA,
                _JINA_MAX_TOKENS, _JINA_FIT_DOC_TOKENS, _JINA_TEST_DOC_TOKENS,
                train_identity, test_identity, _jina_runtime()]
    identity = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()
    directory = Path(run_dir) / 'jina' / identity[:16]
    directory.mkdir(parents=True, exist_ok=True)
    result_path = directory / 'test_scores.pt'
    with _jina_run_lock(directory / 'run.lock'):
        ready = _v43_load(result_path, identity)
        if ready is not None:
            scores = ready['scores']
            if (isinstance(scores, torch.Tensor) and scores.shape == (len(test),)
                    and scores.dtype == torch.float32 and torch.isfinite(scores).all()):
                return scores.numpy()
            raise ValueError(f'Некорректные итоговые оценки Jina в {result_path}.')

        # Use FP32 matmul for the trainable projector and LoRA weights.
        precision = torch.get_float32_matmul_precision()
        try:
            torch.set_float32_matmul_precision('highest')
            tokenizer, model = v43_load_model(deadline)
            groups = v43_groups(tokenizer, train, deadline, max_tokens=_JINA_MAX_TOKENS, max_doc_tokens=_JINA_FIT_DOC_TOKENS)
            _jina_fit(model, groups, directory, identity, deadline)
            del groups
            groups = v43_groups(tokenizer, test, deadline, max_tokens=_JINA_MAX_TOKENS, max_doc_tokens=_JINA_TEST_DOC_TOKENS)
            two = _jina_predict(model, groups, len(test), directory, identity, deadline)
            scores = _jina_predict(model, groups, len(test), directory, identity, deadline, baseline=two)
            _v43_save(dict(signature=identity, scores=torch.from_numpy(scores)), result_path)
            return scores
        finally:
            torch.set_float32_matmul_precision(precision)


## Обучение и запись итоговых рангов

In [ ]:
def reproduce_v47():
    data, work = release_paths()
    with release_lock(work):
        environment = release_environment()
        train, test, sample = upstream_read_inputs(data)
        print("Данные", data, "\nРабочий каталог", work, "\nGPU", environment["gpu"], flush=True)
        oof, test_signals = run_release_upstream(data, work / "upstream", inputs=(train, test, sample))
        baseline, baseline_report = release_v35(oof, test_signals, work_dir=work / "meta")
        del oof, test_signals
        jina = run_jina(train, test, work, float("inf"))
        payload = release_submission(test, sample, baseline, jina)
        target = data / "submission.csv"
        release_atomic(target, payload)
        report = dict(status="completed", rows=len(test), queries=test.query_id.nunique(),
                      csv=str(target), environment=environment, v35=baseline_report,
                      finished_utc=datetime.now(timezone.utc).isoformat())
        release_atomic(work / "run_summary.json", json.dumps(report, ensure_ascii=False, indent=2, default=str).encode())
        print("Файл для отправки:", target)
        return target

submission_path = reproduce_v47()
